# NB03: External validation (H5)

Tests **H5**: CLR-based models (B1, M2) generalise to AusMicrobiome + NGSA holdout.

- **H5 success criterion**: M2 holdout RMSE ≤ 1.1× M2 training RMSE for ≥2 of 4 metals.

AusMicrobiome holdout pipeline:
1. OTU counts (`BASE_16S_OTU.csv.gz`) → genus RA via `BASE_16S_taxonomy.csv`
2. CLR transform (top-200 genera by training-set mean RA, same as NB00)
3. Genus-weighted functional features (same top-N and PCA as NB00)
4. NGSA metal targets (log1p-transformed)
5. CSU mobility features via Spark (`get_csu_mobility_features`)
6. Evaluate B0, B1, M1, M2 with XGBoost NaN routing

**Outputs**
- `data/holdout_results.csv` — transfer RMSE per model × target
- `data/holdout_feature_matrices/AusMicrobiome_NGSA_feature_matrix.parquet`

In [ ]:
import sys
import warnings
warnings.filterwarnings('ignore')

import gzip
import numpy as np
import pandas as pd
from pathlib import Path

for _cand in [Path.cwd() / 'scripts', Path.cwd().parent / 'scripts']:
    if _cand.exists():
        sys.path.insert(0, str(_cand))
        break

DATA_DIR   = next(p for p in [Path.cwd() / 'data', Path.cwd().parent / 'data'] if p.exists())
HOLDOUT_DIR = DATA_DIR / 'holdout_feature_matrices'
HOLDOUT_DIR.mkdir(exist_ok=True)

MICRO_DIR = Path('/home/hmacgregor/BERIL-research-observatory/projects/microbeatlas_metal_ecology/data/aus_microbiome')

try:
    spark
except NameError:
    from berdl_notebook_utils.setup_spark_session import get_spark_session
    spark = get_spark_session()
print('Spark ready:', spark.version)

from modelling import TARGETS, get_features, rmse, fit_final_model, ENV_COLS
from composition_utils import clr_transform, compute_genus_weighted_features, cwm_coverage_fraction
from cwm_utils import load_genus_densities
from env_utils import get_csu_mobility_features

densities = load_genus_densities(genus_trait_path=DATA_DIR / 'genus_trait_table.csv')

## 1. Fit final models on full training set

In [ ]:
feature_matrix = pd.read_parquet(DATA_DIR / 'feature_matrix.parquet')

# Also record which top-200 genera were selected in NB00 (must use same set for holdout CLR)
clr_cols = [c for c in feature_matrix.columns if c.startswith('clr_')]
training_genera = [c.replace('clr_', '') for c in clr_cols]
print(f'Training CLR genera ({len(training_genera)}): {training_genera[:5]}...')

final_models = {}
for target in TARGETS:
    if target not in feature_matrix.columns:
        continue
    for model_name in ['B0', 'B1', 'M1', 'M2']:
        m = fit_final_model(feature_matrix, target, model_name,
                            model_type='xgboost' if model_name != 'B0' else 'xgboost')
        final_models[(target, model_name)] = m
    print(f'  Fit final models for {target}')

print('Final models ready.')

## 2. AusMicrobiome + NGSA holdout

In [ ]:
# ── Taxonomy: OTU → genus ──────────────────────────────────────────────────────
print('Loading AusMicrobiome taxonomy...')
tax = pd.read_csv(MICRO_DIR / 'BASE_16S_taxonomy.csv')
tax.columns = ['OTU_Id', 'genus_str']
tax['genus_lower'] = (
    tax['genus_str'].str.replace('g__', '', regex=False).str.strip().str.lower()
)
tax.loc[
    tax['genus_lower'].isin(['', 'unclassified']) | tax['genus_lower'].isna(),
    'genus_lower',
] = None
otu_to_genus = tax.dropna(subset=['genus_lower']).set_index('OTU_Id')['genus_lower'].to_dict()
print(f'  {len(otu_to_genus):,} OTUs with genus assignment')

# ── OTU counts → genus RA ──────────────────────────────────────────────────────
print('Loading OTU counts...')
with gzip.open(MICRO_DIR / 'BASE_16S_OTU.csv.gz', 'rt') as f:
    otu_counts = pd.read_csv(f, index_col=0)

otu_counts.index = otu_counts.index.map(otu_to_genus)
otu_counts = otu_counts[otu_counts.index.notna()]
genus_counts = otu_counts.groupby(level=0).sum()
genus_counts.index = genus_counts.index.str.lower().str.strip()
genus_ra = genus_counts.T
totals = genus_ra.sum(axis=1).replace(0.0, np.nan)
genus_ra = genus_ra.div(totals, axis=0)
print(f'  Genus RA: {genus_ra.shape[0]:,} samples × {genus_ra.shape[1]:,} genera')

# ── Join with NGSA ─────────────────────────────────────────────────────────────
ngsa = pd.read_csv(MICRO_DIR / 'aus_sample_ngsa.csv')
ngsa['otu_key'] = ngsa['Sample_ID'].str.split('/').str[-1]
ngsa = ngsa.set_index('otu_key')

common = genus_ra.index.intersection(ngsa.index)
print(f'  Common samples: {len(common)}')
genus_ra_c = genus_ra.loc[common]
ngsa_c = ngsa.loc[common]

# ── CLR transform (restricted to training genera) ──────────────────────────────
# Use only genera present in training set to ensure same feature space
aus_top = genus_ra_c.reindex(columns=training_genera, fill_value=0.0)
aus_clr = clr_transform(aus_top)
print(f'  AusMicrobiome CLR: {aus_clr.shape}')

# ── Genus-weighted features ────────────────────────────────────────────────────
aus_gw = compute_genus_weighted_features(
    genus_ra_c, densities, top_n_per_cat=20, n_pca=10,
)
# Align to training GW columns
gw_cols = [c for c in feature_matrix.columns if c.startswith('gw_')]
aus_gw = aus_gw.reindex(columns=gw_cols, fill_value=0.0)
print(f'  AusMicrobiome GW features: {aus_gw.shape}')

# ── CWM coverage ──────────────────────────────────────────────────────────────
coverage = cwm_coverage_fraction(genus_ra_c, densities)
print(f'  Coverage: {coverage.mean():.3f} mean, {(coverage < 0.70).mean():.1%} flagged')

# ── CSU mobility via Spark ────────────────────────────────────────────────────
print('Getting CSU mobility features via Spark...')
aus_coords = pd.DataFrame({
    'lat': pd.to_numeric(ngsa_c['latitude'], errors='coerce').values,
    'lon': pd.to_numeric(ngsa_c['longitude'], errors='coerce').values,
}, index=common)
aus_coords.index.name = 'sample_id'
mob_df = get_csu_mobility_features(spark, aus_coords)
print(f'  CSU matched: {mob_df.dropna(how="all").shape[0]:,} / {len(common):,}')

# ── Build holdout feature matrix ──────────────────────────────────────────────
aus_fm = aus_clr.copy()
for col in aus_gw.columns:
    aus_fm[col] = aus_gw[col].values

# Env features: mob_* from Spark; everything else NaN (XGBoost handles via NaN routing)
for col in ENV_COLS:
    aus_fm[col] = np.nan
for col in ['mob_cu', 'mob_pb', 'mob_as', 'mob_cd', 'mob_cr', 'mob_hg']:
    if col in mob_df.columns:
        aus_fm[col] = mob_df[col].values

# NGSA metal targets
for src, dst in [('ngsa_Cu_ppm','log_Cu_ppm'), ('ngsa_Zn_ppm','log_Zn_ppm'),
                 ('ngsa_Pb_ppm','log_Pb_ppm'), ('ngsa_Ni_ppm','log_Ni_ppm')]:
    if src in ngsa_c.columns:
        aus_fm[dst] = np.log1p(pd.to_numeric(ngsa_c[src], errors='coerce').values)

aus_fm['lat'] = aus_coords['lat'].values
aus_fm['lon'] = aus_coords['lon'].values
aus_fm.to_parquet(HOLDOUT_DIR / 'AusMicrobiome_NGSA_feature_matrix.parquet')

feat_avail = {col: int(aus_fm[col].notna().sum()) for col in ENV_COLS if col in aus_fm.columns}
print('Feature availability:', feat_avail)

## 3. Evaluate final models on AusMicrobiome holdout

In [ ]:
aus_records = []
for target in TARGETS:
    if target not in aus_fm.columns:
        continue
    y = aus_fm[target]
    valid_y = y.notna()

    for model_name in ['B0', 'B1', 'M1', 'M2']:
        model = final_models.get((target, model_name))
        if model is None:
            continue

        if model_name == 'B0':
            n = int(valid_y.sum())
            if n < 5:
                continue
            aus_records.append({
                'holdout': 'AusMicrobiome_NGSA', 'model': model_name,
                'target': target, 'n': n,
                'rmse': rmse(y[valid_y].values, np.full(n, model)),
            })
        else:
            X = get_features(aus_fm, model_name)
            # Ensure column order matches training (XGBoost validates feature_names order)
            train_cols = get_features(feature_matrix.iloc[:0], model_name).columns
            X = X.reindex(columns=train_cols)
            valid = valid_y & ~X.isna().all(axis=1)
            n = int(valid.sum())
            if n < 5:
                continue
            preds = model.predict(X[valid])
            aus_records.append({
                'holdout': 'AusMicrobiome_NGSA', 'model': model_name,
                'target': target, 'n': n,
                'rmse': rmse(y[valid].values, preds),
            })

aus_results = pd.DataFrame(aus_records)
aus_results.to_csv(DATA_DIR / 'holdout_results.csv', index=False)
pivot = aus_results.pivot_table(index='model', columns='target', values='rmse').round(4)
print('AusMicrobiome+NGSA holdout RMSE:')
print(pivot)

## 4. H5: Transfer ratio (holdout / training RMSE)

In [ ]:
cv_all = pd.concat([
    pd.read_csv(DATA_DIR / 'cv_results_baselines.csv'),
    pd.read_csv(DATA_DIR / 'cv_results_models.csv'),
], ignore_index=True)
train_rmse = cv_all[cv_all['model'] == 'M2'].groupby('target')['rmse'].mean()
holdout_rmse = aus_results[aus_results['model'] == 'M2'].set_index('target')['rmse']

ratio = (holdout_rmse / train_rmse).rename('holdout/train_ratio')
h5_pass = ratio <= 1.1
print('H5: M2 transfer ratio (holdout RMSE / training RMSE):')
print(pd.DataFrame({'train_rmse': train_rmse, 'holdout_rmse': holdout_rmse,
                    'ratio': ratio, 'pass (<=1.1)': h5_pass}).round(3))
n_h5 = h5_pass.sum()
print(f'\nH5 OUTCOME: {"SUPPORTED" if n_h5 >= 2 else "NOT SUPPORTED"} ({n_h5}/4 metals ≤ 1.1×)')